# SPR-04 — Figure Generation (scikit-plot)

**Ticket:** SPR-04
**Owner:** Prithila
**Depends on:** SPR-03 (trained models, read-only)
**Folder:** `results/figures/`
**Output:** confusion matrices, ROC curves, precision-recall curves as saved image files

**Note:** this notebook only reads the models saved by SPR-03. It does not retrain anything, so it can run in parallel with SPR-05 (SHAP) without any file conflicts.

## 0. Hugging Face setup: login and download data/models
Run this once per Colab session. It authenticates against the `Sakhiur/signal` repo and pulls the CSVs used below. Add your token in Colab's left sidebar (key icon) under the name `HF_TOKEN` before running this cell.

In [ ]:
!pip install -q huggingface_hub

import os
from google.colab import userdata
from huggingface_hub import login, hf_hub_download

login(token=userdata.get('HF_TOKEN'))

REPO_ID = 'Sakhiur/signal'

os.makedirs('data/interim', exist_ok=True)
os.makedirs('data/processed', exist_ok=True)

hf_files = {
    'X_test_Sakhiur.csv': 'data/interim/X_test.csv',
    'X_train_resampled_Sakhiur.csv': 'data/interim/X_train_resampled.csv',
    'y_test_Sakhiur.csv': 'data/interim/y_test.csv',
    'y_train_resampled_Sakhiur.csv': 'data/interim/y_train_resampled.csv',
}

for repo_filename, local_path in hf_files.items():
    downloaded = hf_hub_download(
        repo_id=REPO_ID,
        repo_type='dataset',
        filename=repo_filename,
    )
    import shutil
    shutil.copy(downloaded, local_path)
    print(f'{repo_filename} -> {local_path}')

print('All data files ready.')

In [ ]:
from huggingface_hub import snapshot_download

models_dir = snapshot_download(
    repo_id=REPO_ID,
    repo_type='dataset',
    allow_patterns='models/*',
    local_dir='.',
)
print('Models downloaded to ./models/')

## 1. Load trained models and test set (read-only)

In [ ]:
import pandas as pd
import numpy as np
import joblib
import os
import matplotlib.pyplot as plt

X_test = pd.read_csv('data/interim/X_test.csv')
y_test = pd.read_csv('data/interim/y_test.csv').iloc[:, 0]

models = {
    'Decision Tree': joblib.load('models/decision_tree/model.joblib'),
    'Random Forest': joblib.load('models/random_forest/model.joblib'),
    'XGBoost': joblib.load('models/xgboost/model.joblib'),
    'Logistic Regression': joblib.load('models/logistic_regression/model.joblib'),
    'Voting Ensemble (soft)': joblib.load('models/voting_ensemble/soft_voting_top3.joblib'),
}

os.makedirs('results/figures', exist_ok=True)
print('Loaded', len(models), 'models')

## 2. Install scikit-plot (if not already available)

In [ ]:
!pip install scikit-plot -q

## 3. Confusion matrices
One figure per model, saved separately so they can be arranged into a collage later in the report's Experimental Results section.

In [ ]:
import scikitplot as skplt

for name, model in models.items():
    y_pred = model.predict(X_test)
    fig, ax = plt.subplots(figsize=(6, 5))
    skplt.metrics.plot_confusion_matrix(y_test, y_pred, ax=ax, normalize=False)
    ax.set_title(f'Confusion Matrix: {name}')
    fname = name.lower().replace(' ', '_').replace('(', '').replace(')', '')
    plt.tight_layout()
    plt.savefig(f'results/figures/confusion_matrix_{fname}.png', dpi=200)
    plt.show()
    plt.close()

## 4. ROC curves (one-vs-rest, multi-class)
`scikit-plot` handles multi-class ROC natively via `predict_proba`. Models without `predict_proba` (e.g. hard-voting ensembles) are skipped here.

In [ ]:
for name, model in models.items():
    if not hasattr(model, 'predict_proba'):
        print(f'Skipping {name}: no predict_proba')
        continue
    y_probas = model.predict_proba(X_test)
    fig, ax = plt.subplots(figsize=(7, 6))
    skplt.metrics.plot_roc(y_test, y_probas, ax=ax)
    ax.set_title(f'ROC Curve: {name}')
    fname = name.lower().replace(' ', '_').replace('(', '').replace(')', '')
    plt.tight_layout()
    plt.savefig(f'results/figures/roc_curve_{fname}.png', dpi=200)
    plt.show()
    plt.close()

## 5. Precision-Recall curves

In [ ]:
for name, model in models.items():
    if not hasattr(model, 'predict_proba'):
        print(f'Skipping {name}: no predict_proba')
        continue
    y_probas = model.predict_proba(X_test)
    fig, ax = plt.subplots(figsize=(7, 6))
    skplt.metrics.plot_precision_recall(y_test, y_probas, ax=ax)
    ax.set_title(f'Precision-Recall Curve: {name}')
    fname = name.lower().replace(' ', '_').replace('(', '').replace(')', '')
    plt.tight_layout()
    plt.savefig(f'results/figures/precision_recall_{fname}.png', dpi=200)
    plt.show()
    plt.close()

## 6. Feature importance bar chart (tree-based models only)
Not from scikit-plot, but useful alongside it and cheap to add here since it reads the same loaded models.

In [ ]:
import seaborn as sns

for name in ['Decision Tree', 'Random Forest', 'XGBoost']:
    model = models[name]
    importances = model.feature_importances_
    feat_df = pd.DataFrame({
        'feature': X_test.columns,
        'importance': importances
    }).sort_values('importance', ascending=False).head(15)

    plt.figure(figsize=(8, 6))
    sns.barplot(data=feat_df, x='importance', y='feature')
    plt.title(f'Top 15 Feature Importances: {name}')
    plt.tight_layout()
    fname = name.lower().replace(' ', '_')
    plt.savefig(f'results/figures/feature_importance_{fname}.png', dpi=200)
    plt.show()
    plt.close()

## 7. Summary
All figures saved to `results/figures/`. This folder feeds directly into SPR-08c (Report: Experimental Results).